# W9 Demo 2 - Three Reinforcement Learning Methods in a Vacuum World

This notebook solves the same small vacuum-cleaner world using three approaches from the Week 9 lecture:

1. Adaptive Dynamic Programming (ADP), a model-based method.
2. Action-utility learning, shown with Q-learning.
3. Policy search, which directly searches over policies.

The point is not to build a large simulator. The point is to keep the environment small enough that the differences among the methods are visible.



## Vacuum world

There are two rooms, A and B. The agent is in one room. Each room can be dirty or clean.

$State = (agent\_position, dirt\_in\_A, dirt\_in\_B)$

Actions:

- Left: move toward room A.
- Right: move toward room B.
- Suck: clean the current room if it is dirty.
- NoOp: do nothing.

Rewards:

- Suck on a dirty square: +10.
- Any non-terminal step has a small cost: -1.
- Bumping into the same side or sucking a clean square is allowed but not useful.
- When both rooms are clean, the state is terminal.



In [1]:
# Import Counter để đếm tần suất, defaultdict để tạo dictionary có giá trị mặc định
from collections import Counter, defaultdict

# Import product để tạo tích Descartes, hữu ích khi sinh tất cả tổ hợp trạng thái
from itertools import product

# Import random để mô phỏng yếu tố ngẫu nhiên trong môi trường
import random

# Import statistics để tính các thống kê như mean, median nếu cần
import statistics


# Hai vị trí có thể có trong môi trường hút bụi
POSITIONS = ("A", "B")

# Các hành động mà agent có thể thực hiện
ACTIONS = ("Left", "Right", "Suck", "NoOp")

# Hệ số chiết khấu cho phần thưởng tương lai
GAMMA = 0.90

# Xác suất bị trượt, tức là hành động di chuyển có thể không thành công
SLIP_PROBABILITY = 0.10

# Số bước tối đa trong một episode
MAX_STEPS = 20


# Tạo toàn bộ không gian trạng thái
# Mỗi state có dạng: (position, dirt_a, dirt_b)
# position: vị trí hiện tại của agent, A hoặc B
# dirt_a: trạng thái bụi ở ô A, 1 là bẩn, 0 là sạch
# dirt_b: trạng thái bụi ở ô B, 1 là bẩn, 0 là sạch
STATES = tuple(
    (position, dirt_a, dirt_b)
    for position in POSITIONS
    for dirt_a in (0, 1)
    for dirt_b in (0, 1)
)

# Lọc ra các trạng thái chưa kết thúc
# Trạng thái kết thúc là khi cả A và B đều sạch
NONTERMINAL_STATES = tuple(
    state for state in STATES
    if not (state[1] == 0 and state[2] == 0)
)


# Kiểm tra một trạng thái có phải terminal hay không
def is_terminal(state):
    # Tách state thành vị trí hiện tại, trạng thái bụi ở A và trạng thái bụi ở B
    _, dirt_a, dirt_b = state

    # Terminal khi cả hai ô A và B đều sạch
    return dirt_a == 0 and dirt_b == 0


# Khởi tạo trạng thái ban đầu cho một episode
def reset_state(rng, both_dirty=False):
    # Chọn ngẫu nhiên vị trí ban đầu của agent
    position = rng.choice(POSITIONS)

    # Nếu both_dirty=True thì cả A và B đều bẩn
    if both_dirty:
        return (position, 1, 1)

    # Nếu không, chọn ngẫu nhiên một trong các cấu hình có ít nhất một ô bẩn
    dirt_a, dirt_b = rng.choice([(1, 1), (1, 0), (0, 1)])

    # Trả về trạng thái ban đầu
    return (position, dirt_a, dirt_b)


# Mô phỏng một bước hành động trong môi trường hút bụi
def vacuum_step(state, action, rng):
    # Tách state thành vị trí hiện tại và trạng thái bụi ở hai ô
    position, dirt_a, dirt_b = state

    # Nếu đã ở trạng thái terminal thì không thay đổi state và reward bằng 0
    if is_terminal(state):
        return state, 0.0

    # Mặc định mỗi hành động tốn chi phí -1
    reward = -1.0

    # Nếu hành động là hút bụi
    if action == "Suck":
        # Nếu agent đang ở A và A bẩn thì hút sạch A, nhận reward dương
        if position == "A" and dirt_a:
            dirt_a = 0
            reward = 10.0

        # Nếu agent đang ở B và B bẩn thì hút sạch B, nhận reward dương
        elif position == "B" and dirt_b:
            dirt_b = 0
            reward = 10.0

        # Nếu hút ở ô đã sạch thì bị phạt
        else:
            reward = -2.0

    # Nếu hành động là đi sang trái
    elif action == "Left":
        # Với xác suất 1 - SLIP_PROBABILITY, hành động thành công và agent đến A
        if rng.random() >= SLIP_PROBABILITY:
            position = "A"

    # Nếu hành động là đi sang phải
    elif action == "Right":
        # Với xác suất 1 - SLIP_PROBABILITY, hành động thành công và agent đến B
        if rng.random() >= SLIP_PROBABILITY:
            position = "B"

    # Nếu không làm gì thì vẫn chịu chi phí bước đi
    elif action == "NoOp":
        reward = -1.0

    # Nếu action không hợp lệ thì báo lỗi
    else:
        raise ValueError(f"Unknown action: {action}")

    # Trả về trạng thái mới và phần thưởng nhận được
    return (position, dirt_a, dirt_b), reward


# Chuyển state sang dạng chuỗi dễ đọc
def state_label(state):
    # Tách state thành vị trí hiện tại và trạng thái bụi
    position, dirt_a, dirt_b = state

    # Trả về mô tả trạng thái:
    # dirty nếu ô bẩn, clean nếu ô sạch
    return f"pos={position}, A={'dirty' if dirt_a else 'clean'}, B={'dirty' if dirt_b else 'clean'}"


# In policy ra màn hình theo từng trạng thái chưa kết thúc
def print_policy(policy, title="Policy"):
    # In tiêu đề của policy
    print(title)

    # Duyệt qua tất cả trạng thái non-terminal
    for state in NONTERMINAL_STATES:
        # In trạng thái và hành động mà policy chọn
        # Nếu state không có trong policy thì in dấu '?'
        print(f"  {state_label(state):40s} -> {policy.get(state, '?')}")


# In tổng số trạng thái trong môi trường
print("Number of states:", len(STATES))

# In số trạng thái chưa kết thúc
print("Number of non-terminal states:", len(NONTERMINAL_STATES))

# In danh sách hành động có thể thực hiện
print("Actions:", ACTIONS)

# In ví dụ mô tả một trạng thái cụ thể
print("Example state:", state_label(("A", 1, 0)))

Number of states: 8
Number of non-terminal states: 6
Actions: ('Left', 'Right', 'Suck', 'NoOp')
Example state: pos=A, A=dirty, B=clean


## Common evaluation helper

All three methods will be evaluated using the same simulator and the same discounted return formula.



In [2]:
# Chạy một episode trong môi trường hút bụi theo policy cho trước
def run_episode(policy, start_state=("A", 1, 1), seed=0, max_steps=MAX_STEPS, return_trace=False):
    # Tạo bộ sinh số ngẫu nhiên với seed cố định
    # để kết quả có thể tái lập
    rng = random.Random(seed)

    # Gán trạng thái ban đầu
    state = start_state

    # trace lưu lại toàn bộ các bước đi trong episode
    trace = []

    # total lưu tổng phần thưởng có chiết khấu
    total = 0.0

    # discount là trọng số chiết khấu cho reward tại mỗi bước
    discount = 1.0

    # Lặp tối đa max_steps bước
    for step_index in range(max_steps):
        # Nếu môi trường đã ở trạng thái terminal thì dừng episode
        if is_terminal(state):
            break

        # Chọn hành động theo policy hiện tại
        action = policy(state)

        # Thực hiện hành động trong môi trường,
        # nhận về trạng thái kế tiếp và phần thưởng
        next_state, reward = vacuum_step(state, action, rng)

        # Lưu lại thông tin của bước hiện tại:
        # state hiện tại, action, reward, và next_state
        trace.append((state, action, reward, next_state))

        # Cộng reward đã chiết khấu vào tổng return
        total += discount * reward

        # Cập nhật hệ số chiết khấu cho bước tiếp theo
        discount *= GAMMA

        # Chuyển sang trạng thái kế tiếp
        state = next_state

    # Nếu cần trả về cả quá trình chạy chi tiết,
    # trả về total và trace
    if return_trace:
        return total, trace

    # Nếu không, chỉ trả về tổng return
    return total


# Đánh giá một policy bằng cách chạy nhiều episode
# và lấy trung bình return thu được
def evaluate_policy(policy, episodes=100, seed=123, both_dirty=True):
    # Tạo bộ sinh số ngẫu nhiên để sinh trạng thái bắt đầu
    rng = random.Random(seed)

    # Danh sách lưu return của từng episode
    returns = []

    # Chạy nhiều episode để ước lượng hiệu quả trung bình của policy
    for episode in range(episodes):
        # Sinh trạng thái bắt đầu ngẫu nhiên
        # Nếu both_dirty=True thì cả hai ô A và B đều bẩn
        start = reset_state(rng, both_dirty=both_dirty)

        # Chạy episode từ trạng thái start,
        # dùng seed khác nhau cho từng episode
        returns.append(run_episode(policy, start_state=start, seed=seed + episode))

    # Trả về mean return của các episode
    return statistics.mean(returns)


# Policy tham khảo được thiết kế thủ công
def greedy_reference_policy(state):
    # Tách state thành vị trí hiện tại và trạng thái bụi ở A, B
    position, dirt_a, dirt_b = state

    # Nếu agent đang ở A và A bẩn thì hút bụi
    if position == "A" and dirt_a:
        return "Suck"

    # Nếu agent đang ở B và B bẩn thì hút bụi
    if position == "B" and dirt_b:
        return "Suck"

    # Nếu A còn bẩn thì di chuyển sang trái để đến A
    if dirt_a:
        return "Left"

    # Nếu B còn bẩn thì di chuyển sang phải để đến B
    if dirt_b:
        return "Right"

    # Nếu không còn bụi thì không làm gì
    return "NoOp"


# In policy tham khảo cho tất cả trạng thái chưa kết thúc
print_policy(
    {state: greedy_reference_policy(state) for state in NONTERMINAL_STATES},
    "Human-designed reference policy"
)

# Đánh giá policy tham khảo bằng mean return
# từ các trạng thái bắt đầu ngẫu nhiên có bụi
print("Mean return from random dirty starts:", round(evaluate_policy(greedy_reference_policy), 3))

Human-designed reference policy
  pos=A, A=clean, B=dirty                  -> Right
  pos=A, A=dirty, B=clean                  -> Suck
  pos=A, A=dirty, B=dirty                  -> Suck
  pos=B, A=clean, B=dirty                  -> Suck
  pos=B, A=dirty, B=clean                  -> Left
  pos=B, A=dirty, B=dirty                  -> Suck
Mean return from random dirty starts: 16.926


# Part 1 - Adaptive Dynamic Programming (ADP)

ADP is model-based. The agent learns:

- a transition model P(s' | s, a) from observed transition counts;
- a reward model R(s, a, s') from observed rewards;
- a value function by solving the learned MDP with value iteration.

This mirrors the lecture idea: learn the model first, then use Bellman backups to plan.



In [3]:
# Chọn hành động theo kiểu epsilon-greedy dựa trên policy hiện tại
def choose_epsilon_greedy_from_policy(state, policy, epsilon, rng):
    # Với xác suất epsilon, hoặc nếu state chưa có trong policy,
    # chọn ngẫu nhiên một hành động để exploration
    if rng.random() < epsilon or state not in policy:
        return rng.choice(ACTIONS)

    # Ngược lại, chọn hành động tốt nhất hiện tại theo policy
    return policy[state]


# Xây dựng mô hình môi trường từ dữ liệu đã quan sát
def build_model(transition_counts, reward_sums):
    # transition_model lưu xác suất chuyển trạng thái:
    # P(next_state | state, action)
    transition_model = {}

    # reward_model lưu reward trung bình:
    # R(state, action, next_state)
    reward_model = {}

    # Duyệt qua từng cặp (state, action) đã được quan sát
    for state_action, next_counter in transition_counts.items():
        # Tổng số lần thực hiện cặp (state, action)
        total = sum(next_counter.values())

        # Tính xác suất chuyển đến từng next_state
        # bằng số lần chuyển đến next_state chia cho tổng số lần quan sát
        transition_model[state_action] = {
            next_state: count / total
            for next_state, count in next_counter.items()
        }

        # Tính reward trung bình cho từng bộ (state, action, next_state)
        for next_state, count in next_counter.items():
            reward_model[
                (state_action[0], state_action[1], next_state)
            ] = reward_sums[
                (state_action[0], state_action[1], next_state)
            ] / count

    # Trả về mô hình transition và mô hình reward đã học được
    return transition_model, reward_model


# Tính Q-value của một cặp (state, action) dựa trên mô hình đã học
def model_q_value(state, action, values, transition_model, reward_model, gamma=GAMMA):
    # Nếu chưa từng quan sát cặp (state, action),
    # xem như hành động đó tạo self-loop với chi phí nhỏ
    if (state, action) not in transition_model:
        # Agent ở lại state hiện tại và nhận reward -1.0
        return -1.0 + gamma * values[state]

    # Biến lưu tổng giá trị kỳ vọng của hành động
    total = 0.0

    # Duyệt qua các next_state có thể xảy ra theo mô hình transition
    for next_state, probability in transition_model[(state, action)].items():
        # Lấy reward trung bình đã học cho transition này
        reward = reward_model[(state, action, next_state)]

        # Cộng vào Q-value theo công thức:
        # Q(s,a) = Σ P(s'|s,a) * [R(s,a,s') + gamma * V(s')]
        total += probability * (reward + gamma * values[next_state])

    # Trả về Q-value ước lượng từ mô hình
    return total


# Chạy value iteration trên mô hình đã học
def value_iteration_from_model(
    transition_model,
    reward_model,
    gamma=GAMMA,
    tolerance=1e-8,
    max_iterations=500
):
    # Khởi tạo giá trị V(s) của tất cả state bằng 0
    values = {state: 0.0 for state in STATES}

    # Lặp value iteration tối đa max_iterations lần
    for iteration in range(max_iterations):
        # delta đo mức thay đổi lớn nhất của value trong một vòng lặp
        delta = 0.0

        # Tạo bản sao để lưu value mới
        new_values = values.copy()

        # Chỉ cập nhật các trạng thái chưa kết thúc
        for state in NONTERMINAL_STATES:
            # Tính giá trị tốt nhất tại state:
            # V(s) = max_a Q(s,a)
            best = max(
                model_q_value(state, action, values, transition_model, reward_model, gamma)
                for action in ACTIONS
            )

            # Cập nhật value mới cho state
            new_values[state] = best

            # Cập nhật delta để kiểm tra hội tụ
            delta = max(delta, abs(new_values[state] - values[state]))

        # Sau khi quét qua các state, thay values bằng new_values
        values = new_values

        # Nếu thay đổi rất nhỏ thì xem như đã hội tụ
        if delta < tolerance:
            break

    # Sau khi có bảng V(s), tạo greedy policy:
    # tại mỗi state chọn action có Q-value lớn nhất
    policy = {
        state: max(
            ACTIONS,
            key=lambda action: model_q_value(
                state,
                action,
                values,
                transition_model,
                reward_model,
                gamma
            )
        )
        for state in NONTERMINAL_STATES
    }

    # Trả về bảng giá trị, policy tối ưu theo mô hình, và số vòng lặp đã dùng
    return values, policy, iteration + 1


# Huấn luyện agent bằng ADP - Adaptive Dynamic Programming
def train_adp(episodes=300, seed=10):
    # Tạo bộ sinh số ngẫu nhiên để kết quả có thể tái lập
    rng = random.Random(seed)

    # Đếm số lần quan sát transition:
    # transition_counts[(state, action)][next_state]
    transition_counts = defaultdict(Counter)

    # Lưu tổng reward cho từng transition:
    # reward_sums[(state, action, next_state)]
    reward_sums = defaultdict(float)

    # Khởi tạo policy ngẫu nhiên cho các state chưa kết thúc
    policy = {
        state: rng.choice(ACTIONS)
        for state in NONTERMINAL_STATES
    }

    # Lưu điểm đánh giá policy tại một số mốc episode
    scores = []

    # Chạy nhiều episode để thu thập dữ liệu và học mô hình
    for episode in range(episodes):
        # Sinh trạng thái bắt đầu ngẫu nhiên
        state = reset_state(rng)

        # Epsilon giảm dần theo thời gian:
        # ban đầu khám phá nhiều, về sau khai thác nhiều hơn
        epsilon = max(0.05, 0.40 * (1 - episode / episodes))

        # Chạy tối đa MAX_STEPS bước trong một episode
        for _ in range(MAX_STEPS):
            # Nếu đã đến terminal state thì dừng episode
            if is_terminal(state):
                break

            # Chọn hành động theo epsilon-greedy từ policy hiện tại
            action = choose_epsilon_greedy_from_policy(state, policy, epsilon, rng)

            # Thực hiện hành động trong môi trường
            next_state, reward = vacuum_step(state, action, rng)

            # Cập nhật số lần quan sát transition này
            transition_counts[(state, action)][next_state] += 1

            # Cộng dồn reward nhận được cho transition này
            reward_sums[(state, action, next_state)] += reward

            # Chuyển sang trạng thái kế tiếp
            state = next_state

        # Sau mỗi episode, xây dựng lại mô hình từ dữ liệu đã thu thập
        transition_model, reward_model = build_model(transition_counts, reward_sums)

        # Chạy value iteration trên mô hình đã học để cải thiện policy
        values, policy, _ = value_iteration_from_model(transition_model, reward_model)

        # Lưu checkpoint để theo dõi quá trình học
        if episode in {0, 1, 2, 9, 49, 99, episodes - 1}:
            scores.append((
                episode + 1,
                evaluate_policy(
                    lambda s, p=policy: p.get(s, "NoOp"),
                    episodes=50,
                    seed=900
                )
            ))

    # Sau khi huấn luyện xong, xây dựng mô hình cuối cùng
    transition_model, reward_model = build_model(transition_counts, reward_sums)

    # Chạy value iteration lần cuối trên mô hình đã học
    values, policy, sweeps = value_iteration_from_model(transition_model, reward_model)

    # Trả về kết quả học được
    return (
        values,
        policy,
        transition_model,
        reward_model,
        transition_counts,
        scores,
        sweeps
    )


# Huấn luyện agent bằng ADP
adp_values, adp_policy, adp_T, adp_R, adp_counts, adp_scores, adp_sweeps = train_adp()

# In các mốc điểm học của ADP
print("ADP learning curve checkpoints:")

for episode, score in adp_scores:
    # In mean return của policy sau một số episode nhất định
    print(f"  after {episode:>3} episodes: mean return = {score:+.3f}")

# In số vòng value iteration cuối cùng trên mô hình đã học
print(f"\nFinal value-iteration sweeps on learned model: {adp_sweeps}")

# In greedy policy học được từ mô hình ADP
print_policy(adp_policy, "ADP greedy policy from the learned model")

# Đánh giá mean return cuối cùng của policy ADP
print("ADP mean return:", round(evaluate_policy(lambda s: adp_policy.get(s, "NoOp")), 3))

ADP learning curve checkpoints:
  after   1 episodes: mean return = -8.784
  after   2 episodes: mean return = +15.621
  after   3 episodes: mean return = +15.621
  after  10 episodes: mean return = +15.621
  after  50 episodes: mean return = +15.621
  after 100 episodes: mean return = +16.973
  after 300 episodes: mean return = +16.973

Final value-iteration sweeps on learned model: 13
ADP greedy policy from the learned model
  pos=A, A=clean, B=dirty                  -> Right
  pos=A, A=dirty, B=clean                  -> Suck
  pos=A, A=dirty, B=dirty                  -> Suck
  pos=B, A=clean, B=dirty                  -> Suck
  pos=B, A=dirty, B=clean                  -> Left
  pos=B, A=dirty, B=dirty                  -> Suck
ADP mean return: 16.926


In [4]:
# In tiêu đề cho phần hiển thị một số xác suất chuyển trạng thái đã học được
print("A few learned transition probabilities:")

# Duyệt qua một vài cặp (state, action) trong mô hình chuyển trạng thái đã học
# sorted(..., key=str) giúp sắp xếp key theo dạng chuỗi để kết quả in ra ổn định
# [:8] chỉ lấy 8 transition đầu tiên để minh họa
for key in sorted(adp_T.keys(), key=str)[:8]:
    # Mỗi key trong adp_T có dạng (state, action)
    state, action = key

    # Tạo chuỗi mô tả các kết quả có thể xảy ra sau khi thực hiện action tại state
    outcomes = ", ".join(
        # Mỗi outcome gồm next_state và xác suất chuyển đến next_state đó
        f"{state_label(next_state)} with P={probability:.2f}"
        for next_state, probability in adp_T[key].items()
    )

    # In transition theo dạng:
    # state hiện tại + action -> các next_state có thể xảy ra kèm xác suất
    print(f"  {state_label(state)} + {action:>5} -> {outcomes}")

A few learned transition probabilities:
  pos=A, A=clean, B=dirty +  Left -> pos=A, A=clean, B=dirty with P=1.00
  pos=A, A=clean, B=dirty +  NoOp -> pos=A, A=clean, B=dirty with P=1.00
  pos=A, A=clean, B=dirty + Right -> pos=B, A=clean, B=dirty with P=0.86, pos=A, A=clean, B=dirty with P=0.14
  pos=A, A=clean, B=dirty +  Suck -> pos=A, A=clean, B=dirty with P=1.00
  pos=A, A=dirty, B=clean +  Left -> pos=A, A=dirty, B=clean with P=1.00
  pos=A, A=dirty, B=clean +  NoOp -> pos=A, A=dirty, B=clean with P=1.00
  pos=A, A=dirty, B=clean + Right -> pos=B, A=dirty, B=clean with P=1.00
  pos=A, A=dirty, B=clean +  Suck -> pos=A, A=clean, B=clean with P=1.00


# Part 2 - Action-Utility Learning with Q-learning

Action-utility learning estimates Q(s, a), the value of taking action a in state s. Q-learning is model-free: it does not learn P(s' | s, a).

The update is:

$Q(s, a) \leftarrow Q(s, a) + \alpha [ r + \gamma ~ max_{a'} Q(s', a') - Q(s, a) ]$



In [5]:
# Huấn luyện agent bằng thuật toán Q-learning
def q_learning(episodes=1200, seed=20, alpha=0.25):
    # Tạo bộ sinh số ngẫu nhiên với seed cố định
    # để kết quả có thể tái lập
    rng = random.Random(seed)

    # q lưu Q-value cho từng cặp (state, action)
    # defaultdict(float) giúp các Q-value chưa có mặc định bằng 0.0
    q = defaultdict(float)

    # scores lưu điểm đánh giá policy tại một số mốc episode
    scores = []

    # Lặp qua nhiều episode để học dần Q-value
    for episode in range(episodes):
        # Sinh trạng thái bắt đầu ngẫu nhiên
        state = reset_state(rng)

        # Epsilon giảm dần theo thời gian:
        # ban đầu exploration nhiều, về sau exploitation nhiều hơn
        epsilon = max(0.03, 0.50 * (1 - episode / episodes))

        # Chạy tối đa MAX_STEPS bước trong mỗi episode
        for _ in range(MAX_STEPS):
            # Nếu đã đến trạng thái terminal thì dừng episode
            if is_terminal(state):
                break

            # Chọn hành động theo epsilon-greedy
            if rng.random() < epsilon:
                # Exploration: chọn hành động ngẫu nhiên
                action = rng.choice(ACTIONS)
            else:
                # Exploitation: chọn hành động có Q-value lớn nhất tại state hiện tại
                action = max(ACTIONS, key=lambda a: q[(state, a)])

            # Thực hiện hành động trong môi trường,
            # nhận về trạng thái kế tiếp và reward
            next_state, reward = vacuum_step(state, action, rng)

            # Nếu next_state là terminal thì không còn giá trị tương lai
            # Ngược lại, lấy Q-value tốt nhất ở next_state
            best_next = (
                0.0
                if is_terminal(next_state)
                else max(q[(next_state, next_action)] for next_action in ACTIONS)
            )

            # TD target của Q-learning:
            # reward hiện tại + gamma * Q tốt nhất ở trạng thái kế tiếp
            td_target = reward + GAMMA * best_next

            # TD error là độ lệch giữa target và Q-value hiện tại
            td_error = td_target - q[(state, action)]

            # Cập nhật Q-value theo công thức:
            # Q(s,a) <- Q(s,a) + alpha * TD_error
            q[(state, action)] += alpha * td_error

            # Chuyển sang trạng thái kế tiếp
            state = next_state

        # Tại một số mốc episode, tạo greedy policy từ Q-value hiện tại
        # rồi đánh giá policy đó bằng mean return
        if episode in {0, 1, 2, 9, 49, 199, 599, episodes - 1}:
            policy = {
                s: max(ACTIONS, key=lambda a: q[(s, a)])
                for s in NONTERMINAL_STATES
            }

            scores.append((
                episode + 1,
                evaluate_policy(
                    lambda s, p=policy: p.get(s, "NoOp"),
                    episodes=50,
                    seed=700
                )
            ))

    # Sau khi học xong, tạo greedy policy cuối cùng từ bảng Q-value
    policy = {
        state: max(ACTIONS, key=lambda action: q[(state, action)])
        for state in NONTERMINAL_STATES
    }

    # Trả về bảng Q-value, policy học được và các checkpoint đánh giá
    return q, policy, scores


# Chạy thuật toán Q-learning
q_values, q_policy, q_scores = q_learning()

# In các checkpoint trong quá trình học
print("Q-learning checkpoints:")

for episode, score in q_scores:
    # In mean return của policy tại từng mốc episode
    print(f"  after {episode:>4} episodes: mean return = {score:+.3f}")

# In greedy policy học được từ Q-learning
print_policy(q_policy, "Q-learning greedy policy")

# Đánh giá mean return cuối cùng của policy học được
print("Q-learning mean return:", round(evaluate_policy(lambda s: q_policy.get(s, "NoOp")), 3))

Q-learning checkpoints:
  after    1 episodes: mean return = -8.784
  after    2 episodes: mean return = +9.582
  after    3 episodes: mean return = +9.582
  after   10 episodes: mean return = +9.582
  after   50 episodes: mean return = +17.009
  after  200 episodes: mean return = +17.009
  after  600 episodes: mean return = +17.009
  after 1200 episodes: mean return = +17.009
Q-learning greedy policy
  pos=A, A=clean, B=dirty                  -> Right
  pos=A, A=dirty, B=clean                  -> Suck
  pos=A, A=dirty, B=dirty                  -> Suck
  pos=B, A=clean, B=dirty                  -> Suck
  pos=B, A=dirty, B=clean                  -> Left
  pos=B, A=dirty, B=dirty                  -> Suck
Q-learning mean return: 16.926


In [6]:
# In tiêu đề cho phần hiển thị Q-values đã học
print("Learned Q-values by state:")

# Duyệt qua từng trạng thái chưa kết thúc
for state in NONTERMINAL_STATES:
    # Tạo danh sách các cặp (action, Q-value) tại state hiện tại
    # Sau đó sắp xếp giảm dần theo Q-value để action tốt nhất đứng đầu
    ranked = sorted(
        ((action, q_values[(state, action)]) for action in ACTIONS),
        key=lambda item: item[1],
        reverse=True
    )

    # Tạo chuỗi ngắn gọn hiển thị Q-value của từng action
    # Ví dụ: Suck:+10.00, Left:+5.20, ...
    compact = ", ".join(
        f"{action}:{value:+.2f}"
        for action, value in ranked
    )

    # In state ở dạng dễ đọc, kèm các action và Q-value tương ứng
    print(f"  {state_label(state):40s} -> {compact}")

Learned Q-values by state:
  pos=A, A=clean, B=dirty                  -> Right:+7.99, NoOp:+6.01, Left:+5.98, Suck:+4.81
  pos=A, A=dirty, B=clean                  -> Suck:+10.00, NoOp:+8.00, Left:+8.00, Right:+6.09
  pos=A, A=dirty, B=dirty                  -> Suck:+17.12, NoOp:+14.27, Right:+14.22, Left:+13.94
  pos=B, A=clean, B=dirty                  -> Suck:+10.00, Right:+8.00, NoOp:+8.00, Left:+5.94
  pos=B, A=dirty, B=clean                  -> Left:+7.97, NoOp:+6.12, Right:+6.10, Suck:+5.09
  pos=B, A=dirty, B=dirty                  -> Suck:+17.09, NoOp:+14.30, Right:+14.27, Left:+14.25


# Part 3 - Policy Search

Policy search skips value functions and transition models. It searches directly in the space of policies.

For this tiny world, a deterministic table policy has only six non-terminal states. That means there are 4^6 = 4096 possible deterministic policies, so we can enumerate them all. This is not scalable, but it makes the policy-search idea very concrete.



In [7]:
# Chuyển một tuple các hành động thành một policy dạng dictionary
# Mỗi hành động trong action_tuple sẽ tương ứng với một state trong NONTERMINAL_STATES
def policy_from_action_tuple(action_tuple):
    # zip ghép từng state với từng action tương ứng
    # Kết quả policy có dạng: {state: action}
    return {
        state: action
        for state, action in zip(NONTERMINAL_STATES, action_tuple)
    }


# Đánh giá một policy được biểu diễn dưới dạng bảng tra cứu
def evaluate_policy_table(policy_table, episodes=80, seed=500):
    # Chuyển policy_table thành một hàm policy
    # Nếu state không có trong bảng thì mặc định chọn "NoOp"
    return evaluate_policy(
        lambda state: policy_table.get(state, "NoOp"),
        episodes=episodes,
        seed=seed
    )


# Tìm kiếm vét cạn tất cả các policy tất định có thể có
def exhaustive_policy_search():
    # Lưu policy tốt nhất tìm được
    best_policy = None

    # Lưu điểm tốt nhất hiện tại
    # Ban đầu đặt là âm vô cùng để bất kỳ policy nào cũng có thể tốt hơn
    best_score = float("-inf")

    # Đếm số policy đã kiểm tra
    checked = 0

    # Lưu các mốc mà tại đó tìm được policy tốt hơn trước đó
    checkpoints = []

    # Sinh tất cả tổ hợp hành động có thể gán cho các state chưa kết thúc
    # repeat=len(NONTERMINAL_STATES) nghĩa là mỗi state sẽ được gán một action
    for action_tuple in product(ACTIONS, repeat=len(NONTERMINAL_STATES)):
        # Tăng số lượng policy đã kiểm tra
        checked += 1

        # Chuyển tuple hành động thành policy dạng dictionary
        policy_table = policy_from_action_tuple(action_tuple)

        # Đánh giá policy hiện tại bằng mean return
        score = evaluate_policy_table(policy_table)

        # Nếu policy hiện tại tốt hơn policy tốt nhất trước đó
        if score > best_score:
            # Cập nhật điểm tốt nhất
            best_score = score

            # Cập nhật policy tốt nhất
            best_policy = policy_table

            # Lưu checkpoint cải thiện
            checkpoints.append((checked, best_score))

    # Trả về policy tốt nhất, điểm tốt nhất,
    # số policy đã kiểm tra và các checkpoint cải thiện
    return best_policy, best_score, checked, checkpoints


# Chạy tìm kiếm vét cạn để tìm policy tốt nhất
search_policy, search_score, checked, search_checkpoints = exhaustive_policy_search()

# In tổng số policy tất định đã được kiểm tra
print(f"Checked {checked} deterministic policies")

# In các mốc mà thuật toán tìm được policy tốt hơn
print("Improvement checkpoints:")

# Chỉ in tối đa 10 checkpoint đầu tiên để tránh output quá dài
for index, score in search_checkpoints[:10]:
    print(f"  policy #{index:>4}: mean return = {score:+.3f}")

# Nếu có nhiều hơn 10 checkpoint thì in dấu "..."
# và thêm 3 checkpoint cuối cùng
if len(search_checkpoints) > 10:
    print("  ...")

    # In 3 lần cải thiện cuối cùng
    for index, score in search_checkpoints[-3:]:
        print(f"  policy #{index:>4}: mean return = {score:+.3f}")


# In policy tốt nhất tìm được bằng direct search
print_policy(search_policy, "Best policy found by direct search")

# In mean return của policy tốt nhất
print("Policy-search mean return:", round(search_score, 3))

Checked 4096 deterministic policies
Improvement checkpoints:
  policy #   1: mean return = -8.784
  policy #   3: mean return = -2.459
  policy #  67: mean return = +1.702
  policy # 131: mean return = +2.216
  policy # 515: mean return = +5.976
  policy # 579: mean return = +15.732
  policy #1699: mean return = +16.945
Best policy found by direct search
  pos=A, A=clean, B=dirty                  -> Right
  pos=A, A=dirty, B=clean                  -> Suck
  pos=A, A=dirty, B=dirty                  -> Suck
  pos=B, A=clean, B=dirty                  -> Suck
  pos=B, A=dirty, B=clean                  -> Left
  pos=B, A=dirty, B=dirty                  -> Suck
Policy-search mean return: 16.945


## Side-by-side comparison

The three agents should discover the same basic behavior:

- If the current room is dirty, Suck.
- If the current room is clean and the other room is dirty, move to the other room.
- If everything is clean, stop.

The methods differ in what they learn:

- ADP learns a model and then plans with Bellman backups.
- Q-learning learns action utilities directly from experience.
- Policy search directly optimizes a policy without explicitly learning utilities.



In [8]:
# Tạo dictionary chứa các phương pháp/policy cần so sánh
methods = {
    # Policy tham khảo được thiết kế thủ công
    "Reference": lambda s: greedy_reference_policy(s),

    # Policy học được từ ADP
    # Nếu state không có trong adp_policy thì chọn mặc định "NoOp"
    "ADP": lambda s: adp_policy.get(s, "NoOp"),

    # Policy học được từ Q-learning
    # Nếu state không có trong q_policy thì chọn mặc định "NoOp"
    "Q-learning": lambda s: q_policy.get(s, "NoOp"),

    # Policy tốt nhất tìm được bằng tìm kiếm trực tiếp
    # Nếu state không có trong search_policy thì chọn mặc định "NoOp"
    "Policy search": lambda s: search_policy.get(s, "NoOp"),
}


# In tiêu đề phần so sánh mean discounted return
print("Mean discounted return from random dirty starts:")

# Duyệt qua từng phương pháp để đánh giá hiệu quả trung bình
for name, policy in methods.items():
    # evaluate_policy chạy nhiều episode từ các trạng thái bẩn ngẫu nhiên
    # episodes=200 giúp lấy trung bình trên 200 lần chạy
    # seed=42 giúp kết quả có thể tái lập
    print(f"  {name:>13}: {evaluate_policy(policy, episodes=200, seed=42):+.3f}")


# In tiêu đề phần trace chi tiết từ cùng một trạng thái bắt đầu
print("\nTrace from start state (A, dirty, dirty):")

# Duyệt qua từng phương pháp để xem đường đi cụ thể của agent
for name, policy in methods.items():
    # Chạy một episode bắt đầu từ:
    # agent ở A, A bẩn, B bẩn
    # return_trace=True để lấy cả tổng return và danh sách các bước đi
    total, trace = run_episode(
        policy,
        start_state=("A", 1, 1),
        seed=99,
        return_trace=True
    )

    # In tên phương pháp và tổng discounted return của episode này
    print(f"\n{name}: return={total:+.3f}")

    # In chi tiết từng bước trong trace
    for state, action, reward, next_state in trace:
        # Mỗi dòng thể hiện:
        # state hiện tại --action/reward--> state kế tiếp
        print(
            f"  {state_label(state):40s} "
            f"--{action:>5}/{reward:+.1f}--> "
            f"{state_label(next_state)}"
        )

Mean discounted return from random dirty starts:
      Reference: +16.966
            ADP: +16.966
     Q-learning: +16.966
  Policy search: +16.966

Trace from start state (A, dirty, dirty):

Reference: return=+17.200
  pos=A, A=dirty, B=dirty                  -- Suck/+10.0--> pos=A, A=clean, B=dirty
  pos=A, A=clean, B=dirty                  --Right/-1.0--> pos=B, A=clean, B=dirty
  pos=B, A=clean, B=dirty                  -- Suck/+10.0--> pos=B, A=clean, B=clean

ADP: return=+17.200
  pos=A, A=dirty, B=dirty                  -- Suck/+10.0--> pos=A, A=clean, B=dirty
  pos=A, A=clean, B=dirty                  --Right/-1.0--> pos=B, A=clean, B=dirty
  pos=B, A=clean, B=dirty                  -- Suck/+10.0--> pos=B, A=clean, B=clean

Q-learning: return=+17.200
  pos=A, A=dirty, B=dirty                  -- Suck/+10.0--> pos=A, A=clean, B=dirty
  pos=A, A=clean, B=dirty                  --Right/-1.0--> pos=B, A=clean, B=dirty
  pos=B, A=clean, B=dirty                  -- Suck/+10.0--> pos

## Takeaways

- ADP is data efficient because each learned transition can be reused in planning, but solving the model can be expensive in large state spaces.
- Q-learning is simple per observation and does not need a model, but it often needs more experience.
- Policy search is conceptually direct: choose a policy, evaluate it, improve it. It becomes difficult when the policy space is large.

